# Preprocessing pipeline

Categorical:
- Encode categories as numbers using LabelEncoder
- 

Numerical:
- Log transform
- Standardscale



In [83]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import sklearn.linear_model as lm

seed = 42
np.random.seed(seed)

sns.set_style('darkgrid')
sns.set_theme(font_scale=1.5)


CATEGORICAL_VARIABLES = ["chd", "famhist"]
CONTINUOUS_VARIABLES = ["sbp", "tobacco", "ldl", "typea", "alcohol", "age"]
INCLUDED_VARIABLES = ["sbp", "tobacco", "ldl", "typea", "alcohol", "age", "chd", "famhist"]

df = pd.read_csv("data/heartDisease.csv")
df = df.drop(labels=["row.names"], axis=1)


X, y = df[INCLUDED_VARIABLES], df["obesity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)


In [84]:

class LogTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self, columns_to_transform):
        self.columns_to_transform = columns_to_transform
    
    def fit(self, X, y=None):
        self.columns_ = X.columns
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.columns_to_transform:
            X[col] = np.log(X[col] + 1/100000)
        return X
    
    def get_feature_names_out(self, *args, **params):
        return self.columns_


In [85]:


num_pipeline = Pipeline(steps=[
        ('log_transform', LogTransformer(["alcohol", "tobacco"])),
        ('scaler', StandardScaler().set_output(transform='pandas'))
])


cat_pipeline = Pipeline(steps=[
        ('onehotencoder', OneHotEncoder())
])

cat_pipeline.fit_transform(X_train.copy())


preproc = ColumnTransformer([
    ("num", num_pipeline, CONTINUOUS_VARIABLES),
    ("cat", cat_pipeline, CATEGORICAL_VARIABLES),
], remainder='passthrough')


preproc.fit(X_train.copy())
X_train_preprocessed = pd.DataFrame(preproc.transform(X_train.copy()), columns=preproc.get_feature_names_out().tolist())

